# Transformer Benchmark on Google Colab (TPU/Tesla T4)

本Notebook用于在Google Colab上实现Transformer模型的训练与评估，支持TPU和Tesla T4 GPU环境。涵盖数据加载、模型定义、训练、评估与可视化等完整流程。

## 1. 导入必要的库

本节导入PyTorch、NumPy、Matplotlib等所需的Python库，并自动检测Colab运行环境（TPU/GPU/CPU）。

In [ ]:
# Install torch_xla for TPU if needed (Colab TPU only)
import os
import sys
import subprocess

try:
    import torch_xla
    TPU_AVAILABLE = True
except ImportError:
    TPU_AVAILABLE = False

# Install torch_xla if on Colab TPU
if 'COLAB_TPU_ADDR' in os.environ and not TPU_AVAILABLE:
    print('Installing torch_xla for TPU...')
    subprocess.run(['pip', 'install', 'torch_xla'], check=True)
    import torch_xla
    TPU_AVAILABLE = True

import torch
import numpy as np
import matplotlib.pyplot as plt

# Device detection
device = 'cpu'
if TPU_AVAILABLE:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print('Using TPU')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print('Using GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device('cpu')
    print('Using CPU')

## 2. 加载和预处理数据

本节将生成合成数据用于基准测试，包括低熵（重复）、中熵（代码样式）、高熵（自然语言样式）三类输入。

In [ ]:
# Generate synthetic data for benchmarking

def generate_synthetic_data(seq_len, dim, regime='high'):
    """
    Generate synthetic data for different entropy regimes.
    regime: 'low' (repetitive), 'medium' (code-like), 'high' (random)
    """
    if regime == 'low':
        # Repetitive pattern
        base = torch.ones(dim) * 0.5
        data = base.repeat(seq_len, 1)
    elif regime == 'medium':
        # Code-like: mix of patterns and randomness
        data = torch.zeros(seq_len, dim)
        for i in range(seq_len):
            if i % 4 == 0:
                data[i] = torch.randn(dim) * 0.1 + 0.5
            else:
                data[i] = torch.randn(dim) * 0.5
    else:
        # High entropy: standard normal
        data = torch.randn(seq_len, dim)
    return data

# Example usage
seq_len = 512
hidden_dim = 1024
low_entropy = generate_synthetic_data(seq_len, hidden_dim, 'low')
medium_entropy = generate_synthetic_data(seq_len, hidden_dim, 'medium')
high_entropy = generate_synthetic_data(seq_len, hidden_dim, 'high')

print('Low entropy sample:', low_entropy[0][:5])
print('Medium entropy sample:', medium_entropy[0][:5])
print('High entropy sample:', high_entropy[0][:5])

## 3. 定义Transformer模型结构

本节使用PyTorch定义一个简化版的Transformer模型结构，包含编码器、解码器和超参数设置。

In [ ]:
import torch.nn as nn

class SimpleTransformer(nn.Module):
    def __init__(self, d_model=1024, nhead=8, num_layers=2, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout
        )
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, src):
        # src: (seq_len, batch, d_model)
        output = self.transformer_encoder(src)
        output = self.fc_out(output)
        return output

# Example instantiation
model = SimpleTransformer(d_model=hidden_dim, nhead=8, num_layers=2).to(device)
print(model)

## 4. 模型训练与验证

本节编写训练循环，对Transformer模型进行训练，并在验证集（此处用合成数据模拟）上评估性能。

In [ ]:
# Training loop (using synthetic data as both input and target for demonstration)
def train_model(model, data, epochs=3, lr=1e-3):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    losses = []
    for epoch in range(epochs):
        # Transformer expects (seq_len, batch, d_model)
        src = data.unsqueeze(1).to(device)  # (seq_len, 1, d_model)
        target = data.unsqueeze(1).to(device)
        optimizer.zero_grad()
        output = model(src)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")
    return losses

# Example: train on high-entropy data
losses = train_model(model, high_entropy, epochs=3)
plt.plot(losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 5. 模型评估与可视化

本节对训练好的模型进行评估，输出损失等指标，并用Matplotlib进行结果可视化。

In [ ]:
# Evaluate model on different entropy regimes
def evaluate_model(model, data, desc):
    model.eval()
    with torch.no_grad():
        src = data.unsqueeze(1).to(device)
        target = data.unsqueeze(1).to(device)
        output = model(src)
        mse = nn.functional.mse_loss(output, target).item()
        print(f"{desc} MSE: {mse:.4f}")
    return mse

mse_low = evaluate_model(model, low_entropy, 'Low entropy')
mse_medium = evaluate_model(model, medium_entropy, 'Medium entropy')
mse_high = evaluate_model(model, high_entropy, 'High entropy')

# Visualize comparison
plt.bar(['Low', 'Medium', 'High'], [mse_low, mse_medium, mse_high])
plt.title('Model MSE on Different Entropy Regimes')
plt.ylabel('MSE')
plt.show()

In [ ]:
# 合并保存结果和硬件类型为一段代码
import json

if TPU_AVAILABLE:
    hardware = 'TPU'
elif torch.cuda.is_available():
    hardware = 'GPU'
else:
    hardware = 'CPU'

results = {
    'mse_low': mse_low,
    'mse_medium': mse_medium,
    'mse_high': mse_high,
    'loss_curve': losses,
    'hardware': hardware
}

with open('colab_transformer_results.json', 'w') as f:
    json.dump(results, f)

print(f"Results saved to colab_transformer_results.json. Hardware: {hardware}")